# IndicMedFacts — experiments notebook

In [ ]:
!pip install -q "numpy<2"


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /dist_home/common-apps/conda/bin/python -m pip install --upgrade pip


In [ ]:
!pip install -q "transformers>=4.41" accelerate scikit-learn pandas openai statsmodels


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /dist_home/common-apps/conda/bin/python -m pip install --upgrade pip


In [ ]:
%pip install pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ---- Part 0.2 · config ------------------------------------------------------
import os, re, json, time, random, zipfile, shutil
from collections import Counter
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, cohen_kappa_score

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

OUT_ROOT = "outputs"
BASE = f"{OUT_ROOT}/baselines"      # results JSONs + per-item preds CSVs
ANA  = f"{OUT_ROOT}/analysis"       # paper-ready tables
ANN  = f"{OUT_ROOT}/annotation"     # Phase-2 kappa-study sheets
for d in (BASE, ANA, ANN):
    os.makedirs(d, exist_ok=True)

SEED = 42
LABELS = ["False", "Mostly False", "Partly True"]
label2id = {l: i for i, l in enumerate(LABELS)}

ENCODERS = {
    "BERT-base":    ("bert-base-uncased",               1e-5),
    "RoBERTa-base": ("roberta-base",                    2e-5),
    "BioBERT":      ("dmis-lab/biobert-base-cased-v1.2", 1e-5),
    "MuRIL":        ("google/muril-base-cased",         1e-5),
}
# DeBERTa-v3-base is EXCLUDED per §5.1b: identical majority-collapse at lr 1e-5/2e-5/3e-5,
# fp16 & fp32, with and without class weights (known HF/torch pathology, not a data issue).

RUN_VANILLA      = True
RUN_SMOKE        = False
RESTORE_ARCHIVED = False


SEEDS_MULTI = [42, 43, 44]
MULTISEED_CONFIGS = [("BERT-base", "claim"), ("BERT-base", "claim_evid"), ("RoBERTa-base", "claim_evid")]

DATA = next((p for p in ["outputs/indicmedfacts_clean.csv", "indicmedfacts_clean.csv",
                         "/content/indicmedfacts_clean.csv"] if os.path.exists(p)), None)
assert DATA, "indicmedfacts_clean.csv not found — upload it next to this notebook."
print("data:", DATA, "| OUT_ROOT:", OUT_ROOT)

if RESTORE_ARCHIVED:
    import glob
    n = 0
    for src in ["results_llm.json", "results_llm_evidence.json", "results_encoders.json"] + \
               [os.path.basename(p) for p in glob.glob("/content/preds_*.csv") + glob.glob("preds_*.csv")]:
        for root in (".", "/content"):
            s = os.path.join(root, src)
            d = os.path.join(BASE, src)
            if os.path.exists(s) and not os.path.exists(d):
                shutil.copy(s, d); n += 1
    print(f"restored {n} archived artefact(s) into {BASE} — matching runs will be skipped")

data: indicmedfacts_clean.csv | OUT_ROOT: outputs


## Part 1 · Data load + sanity audit ✅

In [ ]:
# ---- Part 1 · load + fingerprint --------------------------------------------
df = pd.read_csv(DATA)
assert df["label"].isin(LABELS).all(), "unexpected label values"
df["y"] = df["label"].map(label2id)

train = df[df.split == "train"].copy()
dev   = df[df.split == "dev"].copy()
test  = df[df.split == "test"].copy()
GOLD  = test["y"].values

def _ck(name, got, want, hard=True):
    ok = got == want
    print(f"  [{'OK ' if ok else 'D!!'}] {name}: expected={want}  recomputed={got}")
    if hard and not ok:
        raise AssertionError(f"{name} mismatch — this is not the audited §3 snapshot of indicmedfacts_clean.csv")

print("fingerprint vs lab notebook v4 (audit 2026-07-18):")
_ck("total rows", len(df), 2745)
_ck("splits (train/dev/test)", (len(train), len(dev), len(test)), (1921, 412, 412))
lv = df["label"].value_counts()
_ck("labels (F/MF/PT)", (int(lv["False"]), int(lv["Mostly False"]), int(lv["Partly True"])), (1517, 965, 263))
tv = test["label"].value_counts()
_ck("test labels (F/MF/PT)", (int(tv["False"]), int(tv["Mostly False"]), int(tv["Partly True"])), (228, 145, 39))

# class weights (inverse frequency, train-derived) — §5.1b records 0.603 / 0.949 / 3.480
cnt = train["y"].value_counts().sort_index().values
CLASS_W = (len(train) / (3 * cnt)).round(4)
print("class weights (F/MF/PT):", CLASS_W.tolist(), " (§5.1b: [0.603, 0.949, 3.480])")

# evidence availability — §3: empty doctor_explanation 5 train / 1 dev / 0 test
wc = df["doctor_explanation"].fillna("").astype(str).str.split().str.len()
empt = df[wc == 0]["split"].value_counts()
print("empty doctor_explanation per split:", empt.to_dict(), " (expected {'train': 5, 'dev': 1})")
_ck("empty evidence in TEST", int((test["doctor_explanation"].fillna("").astype(str).str.split().str.len() == 0).sum()), 0)
print("\nsnapshot verified — safe to proceed.")

fingerprint vs lab notebook v4 (audit 2026-07-18):
  [OK ] total rows: expected=2745  recomputed=2745
  [OK ] splits (train/dev/test): expected=(1921, 412, 412)  recomputed=(1921, 412, 412)
  [OK ] labels (F/MF/PT): expected=(1517, 965, 263)  recomputed=(1517, 965, 263)
  [OK ] test labels (F/MF/PT): expected=(228, 145, 39)  recomputed=(228, 145, 39)
class weights (F/MF/PT): [0.603, 0.9486, 3.4801]  (§5.1b: [0.603, 0.949, 3.480])
empty doctor_explanation per split: {'train': 5, 'dev': 1}  (expected {'train': 5, 'dev': 1})
  [OK ] empty evidence in TEST: expected=0  recomputed=0

snapshot verified — safe to proceed.


In [ ]:
%pip install torch transformers

Note: you may need to restart the kernel to use updated packages.


## Part 2 · Shared engine

In [ ]:
# ---- Part 2 · engine ---------------------------------------------------------
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

# --- Part 2 engine ---
import torch
import torch.nn.functional as TF
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          set_seed as hf_set_seed)

print("cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU (Parts 3-5, 12 will be very slow)")

ENC_RESULTS = f"{BASE}/results_encoders.json"

def load_registry(path):
    return json.load(open(path)) if os.path.exists(path) else {}

def save_registry(reg, path):
    json.dump(reg, open(path, "w"), indent=2)

def safe_name(s):
    return re.sub(r"[^A-Za-z0-9_+-]+", "_", s)

def metrics_from_preds(y_true, y_pred):
    per = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2], zero_division=0)
    return {"n": int(len(y_true)),
            "acc": round(float(accuracy_score(y_true, y_pred)), 4),
            "macro_f1": round(float(f1_score(y_true, y_pred, average="macro", zero_division=0)), 4),
            "f1_false": round(float(per[0]), 4),
            "f1_mostly_false": round(float(per[1]), 4),
            "f1_partly_true": round(float(per[2]), 4),
            "cm": confusion_matrix(y_true, y_pred, labels=[0, 1, 2]).tolist()}

FLOOR = metrics_from_preds(GOLD, np.zeros(len(GOLD), dtype=int))   # majority-class predictor
print("majority floor:", {k: FLOOR[k] for k in ("acc", "macro_f1")}, " (§5.1: 0.553 / 0.238)")

def parse_label(text):
    # strict parser — identical priority order to run_llm_icl_v2.py (v2.1)
    t = (text or "").lower()
    if "mostly false" in t or "mostly-false" in t: return 1
    if any(w in t for w in ["partly", "half", "mislead", "mostly true", "partial", "mixed"]): return 2
    if "false" in t: return 0
    if "true" in t: return 2      # no fully-true class; leftover 'true' -> Partly True
    return None                   # unparsed

def multi(x):
    # multi-value cell parser (list-string or | ; , separated) — same as audit_indicmedfacts.py
    if isinstance(x, list): return [str(i).strip() for i in x]
    if x is None or (isinstance(x, float) and np.isnan(x)): return []
    s = str(x).strip()
    if not s or s.lower() in ("nan", "none"): return []
    if s.startswith("["):
        import ast as _ast
        try: return [str(i).strip().strip("'\"") for i in _ast.literal_eval(s)]
        except Exception: pass
    for sep in ("|", ";", ","):
        if sep in s: return [p.strip() for p in s.split(sep) if p.strip()]
    return [s]

def preds_path_for(key):
    # run-key -> per-item predictions CSV (encoder and LLM naming conventions)
    m = re.match(r"(.+) \[(.+)\] seed(\d+)$", key)
    if m:
        return f"{BASE}/preds_enc_{safe_name(m.group(1) + '_' + m.group(2))}_seed{m.group(3)}.csv"
    m = re.match(r"(.+) \[(.+)\]$", key)
    return f"{BASE}/preds_{safe_name(m.group(1) + '_' + m.group(2))}.csv"

def load_pred_vec(key):
    # per-item predictions aligned to the canonical test order (paired stats depend on this)
    p = pd.read_csv(preds_path_for(key)).set_index("row").loc[test.index]
    return p["pred"].map(label2id).values

# the ~6 systems the paper's headline claims rest on (filter by file existence at use time)
HEADLINE_KEYS = [f"BERT-base [claim] seed{SEED}",
                 "GPT-OSS-20B [zero_shot]",
                 f"BERT-base [claim_evid] seed{SEED}",
                 f"RoBERTa-base [claim_evid] seed{SEED}",
                 "Gemma-4-26B-A4B [zero_shot+evid]",
                 "DeepSeek-V4-Pro [few_shot+evid]"]

def texts_for(frame, condition):
    claims = frame["claim_text"].astype(str).tolist()
    evid   = frame["doctor_explanation"].fillna("").astype(str).tolist()
    if condition == "claim":      return claims, None
    if condition == "claim_evid": return claims, evid
    if condition == "evid_only":  return evid, None
    raise ValueError(condition)

class ClsDS(Dataset):
    def __init__(self, enc, labels):
        self.enc, self.labels = enc, list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kw):
        super().__init__(**kw)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float)
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        loss = TF.cross_entropy(outputs.logits, labels,
                                weight=self.class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

def run_encoder(model_key, condition, seed=42, weighted=True):
    # resume-safe: one JSON key + one preds CSV per (model, condition, seed, weighting)
    hf_id, lr = ENCODERS[model_key]
    max_len = 256 if condition == "claim" else 512
    bs, accum = (16, 1) if condition == "claim" else (8, 2)
    tag = condition + ("" if weighted else "_vanilla")
    run_id = f"{model_key} [{tag}] seed{seed}"
    ppath = f"{BASE}/preds_enc_{safe_name(model_key + '_' + tag)}_seed{seed}.csv"
    reg = load_registry(ENC_RESULTS)
    if run_id in reg and os.path.exists(ppath):
        print(f"-- {run_id} already done, skipping")
        return reg[run_id]
    print("#" * 62); print(f"{run_id}   ({hf_id}, lr={lr}, max_len={max_len}, bs={bs}x{accum})"); print("#" * 62)
    hf_set_seed(seed)
    tok = AutoTokenizer.from_pretrained(hf_id)
    a, b = texts_for(train, condition)
    enc_tr = tok(a, b, truncation=True, max_length=max_len) if b is not None else tok(a, truncation=True, max_length=max_len)
    a, b = texts_for(test, condition)
    enc_te = tok(a, b, truncation=True, max_length=max_len) if b is not None else tok(a, truncation=True, max_length=max_len)
    model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=3)
    args = TrainingArguments(
        output_dir=f"tmp_{safe_name(run_id)}", seed=seed, learning_rate=lr,
        num_train_epochs=6, per_device_train_batch_size=bs, gradient_accumulation_steps=accum,
        per_device_eval_batch_size=32, warmup_ratio=0.06, weight_decay=0.01,
        fp16=False, logging_steps=100, save_strategy="no", report_to="none")
    coll = DataCollatorWithPadding(tok)
    if weighted:
        trainer = WeightedTrainer(model=model, args=args, train_dataset=ClsDS(enc_tr, train["y"]),
                                  data_collator=coll, class_weights=CLASS_W)
    else:
        trainer = Trainer(model=model, args=args, train_dataset=ClsDS(enc_tr, train["y"]),
                          data_collator=coll)
    trainer.train()
    preds = np.argmax(trainer.predict(ClsDS(enc_te, test["y"])).predictions, axis=-1)
    res = metrics_from_preds(GOLD, preds)
    res.update({"hf_id": hf_id, "condition": condition, "weighted": weighted, "seed": seed, "lr": lr})
    pd.DataFrame({"row": test.index,
                  "id": test["id"].values if "id" in test.columns else test.index,
                  "gold": test["label"].values,
                  "pred": [LABELS[p] for p in preds]}).to_csv(ppath, index=False)
    reg = load_registry(ENC_RESULTS); reg[run_id] = res; save_registry(reg, ENC_RESULTS)
    print(f"  acc={res['acc']:.3f}  macro-F1={res['macro_f1']:.3f}  F1(F/MF/PT)="
          f"{res['f1_false']:.3f}/{res['f1_mostly_false']:.3f}/{res['f1_partly_true']:.3f}")
    del model, trainer
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return res

def result_table(entries):
    # entries: list of (display_name, metrics_dict)
    rows = []
    for name, r in entries:
        rows.append({"System": name, "Acc": r["acc"], "Macro-F1": r["macro_f1"],
                     "F1 False": r["f1_false"], "F1 MostlyFalse": r["f1_mostly_false"],
                     "F1 PartlyTrue": r["f1_partly_true"]})
    return pd.DataFrame(rows)

cuda: True | NVIDIA RTX 6000 Ada Generation
majority floor: {'acc': 0.5534, 'macro_f1': 0.2375}  (§5.1: 0.553 / 0.238)


In [ ]:
%pip install "accelerate>=1.1.0"

Note: you may need to restart the kernel to use updated packages.


## Part 3 · Encoder baselines — claim-only, class-weighted ✅

In [ ]:
# ---- Part 3 · claim-only weighted runs (~5-8 min each on T4) -----------------
assert len(test) == 412 and int((test["label"] == "False").sum()) == 228, \
    "TRIPWIRE: the frozen split was modified after Part 1 (e.g. a pasted train_test_split) — restore Parts 0-2 and re-run them unedited"
CLAIM_KEYS = []
for mk in ENCODERS:
    run_encoder(mk, "claim", seed=SEED, weighted=True)
    CLAIM_KEYS.append(f"{mk} [claim] seed{SEED}")

reg = load_registry(ENC_RESULTS)
t3 = result_table([("Majority floor", FLOOR)] + [(k, reg[k]) for k in CLAIM_KEYS])
print(t3.to_string(index=False))
t3.to_csv(f"{ANA}/table_encoders_claim.csv", index=False)

##############################################################
BERT-base [claim] seed42   (bert-base-uncased, lr=1e-05, max_len=256, bs=16x1)
##############################################################


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_

Step,Training Loss
100,1.104721
200,1.077053
300,1.016541
400,0.967797
500,0.888292
600,0.807507
700,0.744721


  acc=0.544  macro-F1=0.449  F1(F/MF/PT)=0.643/0.468/0.235
##############################################################
RoBERTa-base [claim] seed42   (roberta-base, lr=2e-05, max_len=256, bs=16x1)
##############################################################


...

  acc=0.544  macro-F1=0.432  F1(F/MF/PT)=0.629/0.501/0.167
##############################################################
MuRIL [claim] seed42   (google/muril-base-cased, lr=1e-05, max_len=256, bs=16x1)
##############################################################


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

Step,Training Loss
100,1.098449
200,1.097790
300,1.090539
400,1.084068
500,1.067903
600,1.052192
700,1.035838


  acc=0.556  macro-F1=0.372  F1(F/MF/PT)=0.675/0.440/0.000
                     System    Acc  Macro-F1  F1 False  F1 MostlyFalse  F1 PartlyTrue
             Majority floor 0.5534    0.2375    0.7125          0.0000         0.0000
   BERT-base [claim] seed42 0.5437    0.4486    0.6429          0.4675         0.2353
RoBERTa-base [claim] seed42 0.5049    0.4032    0.5945          0.4885         0.1266
     BioBERT [claim] seed42 0.5437    0.4325    0.6294          0.5015         0.1667
       MuRIL [claim] seed42 0.5558    0.3716    0.6748          0.4399         0.0000


In [ ]:
# ---- Part 3b · optional §5.1 vanilla diagnostic ------------------------------
assert len(test) == 412 and int((test["label"] == "False").sum()) == 228, \
    "TRIPWIRE: the frozen split was modified after Part 1 (e.g. a pasted train_test_split) — restore Parts 0-2 and re-run them unedited"
if RUN_VANILLA:
    for mk in ENCODERS:
        run_encoder(mk, "claim", seed=SEED, weighted=False)
    reg = load_registry(ENC_RESULTS)
    keys = [f"{mk} [claim_vanilla] seed{SEED}" for mk in ENCODERS]
    tv = result_table([("Majority floor", FLOOR)] + [(k, reg[k]) for k in keys])
    print(tv.to_string(index=False))
    tv.to_csv(f"{ANA}/table_encoders_claim_vanilla.csv", index=False)
else:
    print("skipped (RUN_VANILLA=False) — §5.1 vanilla numbers already recorded in the lab notebook")

##############################################################
BERT-base [claim_vanilla] seed42   (bert-base-uncased, lr=1e-05, max_len=256, bs=16x1)
##############################################################


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_

Step,Training Loss
100,0.940526
200,0.894585
300,0.832748
400,0.808971
500,0.721280
600,0.664652
700,0.595766


  acc=0.570  macro-F1=0.384  F1(F/MF/PT)=0.668/0.485/0.000
##############################################################
RoBERTa-base [claim_vanilla] seed42   (roberta-base, lr=2e-05, max_len=256, bs=16x1)
##############################################################


...

  acc=0.595  macro-F1=0.434  F1(F/MF/PT)=0.691/0.516/0.095
##############################################################
MuRIL [claim_vanilla] seed42   (google/muril-base-cased, lr=1e-05, max_len=256, bs=16x1)
##############################################################


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

Step,Training Loss
100,1.075991
200,1.013276
300,0.970260
400,0.957646
500,0.938895
600,0.941162
700,0.925615


  acc=0.553  macro-F1=0.237  F1(F/MF/PT)=0.713/0.000/0.000
                             System    Acc  Macro-F1  F1 False  F1 MostlyFalse  F1 PartlyTrue
                     Majority floor 0.5534    0.2375    0.7125          0.0000         0.0000
   BERT-base [claim_vanilla] seed42 0.5704    0.3843    0.6680          0.4848         0.0000
RoBERTa-base [claim_vanilla] seed42 0.5777    0.3954    0.6754          0.5108         0.0000
     BioBERT [claim_vanilla] seed42 0.5947    0.4340    0.6907          0.5161         0.0952
       MuRIL [claim_vanilla] seed42 0.5534    0.2375    0.7125          0.0000         0.0000


## Part 4 · Encoder + evidence contrast ✅

In [ ]:
# ---- Part 4 · +evidence runs (~15-25 min each on T4) -------------------------
assert len(test) == 412 and int((test["label"] == "False").sum()) == 228, \
    "TRIPWIRE: the frozen split was modified after Part 1 (e.g. a pasted train_test_split) — restore Parts 0-2 and re-run them unedited"
EVID_KEYS = []
for mk in ENCODERS:
    run_encoder(mk, "claim_evid", seed=SEED, weighted=True)
    EVID_KEYS.append(f"{mk} [claim_evid] seed{SEED}")

reg = load_registry(ENC_RESULTS)
t4 = result_table([("Majority floor", FLOOR)] + [(k, reg[k]) for k in EVID_KEYS])
print(t4.to_string(index=False))
t4.to_csv(f"{ANA}/table_encoders_evid.csv", index=False)

print("\nevidence delta (macro-F1, +evid minus claim-only):")
for mk in ENCODERS:
    d = reg[f"{mk} [claim_evid] seed{SEED}"]["macro_f1"] - reg[f"{mk} [claim] seed{SEED}"]["macro_f1"]
    print(f"  {mk:14s} {d:+.3f}")

##############################################################
BERT-base [claim_evid] seed42   (bert-base-uncased, lr=1e-05, max_len=512, bs=8x2)
##############################################################


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_

Step,Training Loss
100,2.209887
200,2.090828
300,1.851068
400,1.612337
500,1.417381
600,1.233175
700,1.078499


  acc=0.617  macro-F1=0.572  F1(F/MF/PT)=0.689/0.564/0.463
##############################################################
RoBERTa-base [claim_evid] seed42   (roberta-base, lr=2e-05, max_len=512, bs=8x2)
##############################################################


...

  acc=0.612  macro-F1=0.551  F1(F/MF/PT)=0.696/0.547/0.410
##############################################################
MuRIL [claim_evid] seed42   (google/muril-base-cased, lr=1e-05, max_len=512, bs=8x2)
##############################################################


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

Step,Training Loss
100,2.193963
200,2.178299
300,2.165113
400,2.160170
500,2.129264
600,2.132512
700,2.092340


  acc=0.570  macro-F1=0.398  F1(F/MF/PT)=0.647/0.546/0.000
                          System    Acc  Macro-F1  F1 False  F1 MostlyFalse  F1 PartlyTrue
                  Majority floor 0.5534    0.2375    0.7125          0.0000         0.0000
   BERT-base [claim_evid] seed42 0.6165    0.5721    0.6893          0.5636         0.4634
RoBERTa-base [claim_evid] seed42 0.6845    0.5768    0.7817          0.6039         0.3448
     BioBERT [claim_evid] seed42 0.6117    0.5512    0.6963          0.5472         0.4103
       MuRIL [claim_evid] seed42 0.5704    0.3978    0.6471          0.5464         0.0000

evidence delta (macro-F1, +evid minus claim-only):
  BERT-base      +0.124
  RoBERTa-base   +0.174
  BioBERT        +0.119
  MuRIL          +0.026


## Part 5 · Leakage probes ⚠ → ✅

In [ ]:
# ---- Part 5 · keyword probes + explanation-only encoders ---------------------
assert len(test) == 412 and int((test["label"] == "False").sum()) == 228, \
    "TRIPWIRE: the frozen split was modified after Part 1 (e.g. a pasted train_test_split) — restore Parts 0-2 and re-run them unedited"
# (a) keyword rule on test explanations — same strict parser as LLM scoring,
#     unparsed -> majority class (protocol-consistent canonical probe)
kw_preds = np.array([(parse_label(str(x)) if parse_label(str(x)) is not None else 0)
                     for x in test["doctor_explanation"].fillna("")])
kw = metrics_from_preds(GOLD, kw_preds)
reg = load_registry(ENC_RESULTS); reg["Keyword probe [explanation]"] = kw; save_registry(reg, ENC_RESULTS)
pd.DataFrame({"row": test.index, "gold": test["label"].values,
              "pred": [LABELS[p] for p in kw_preds]}).to_csv(f"{BASE}/preds_enc_keyword_probe.csv", index=False)
print(f"keyword probe [doctor_explanation]: acc={kw['acc']:.3f} macro-F1={kw['macro_f1']:.3f}")
print("  (§5.3's archived 0.481/0.304 used the lost probe script's slightly different rule;")
print("   all 24 swept variants give macro-F1 0.25-0.34 -> conclusion is rule-independent)")

# (b) quick_take is near-verbatim leakage -> excluded from every input condition
qk_preds = np.array([(parse_label(str(x)) if parse_label(str(x)) is not None else 0)
                     for x in test["quick_take"].fillna("")])
qk = metrics_from_preds(GOLD, qk_preds)
reg = load_registry(ENC_RESULTS); reg["Keyword probe [quick_take]"] = qk; save_registry(reg, ENC_RESULTS)
print(f"keyword probe [quick_take]:         acc={qk['acc']:.3f} macro-F1={qk['macro_f1']:.3f}"
      "   <- the verdict in disguise; this is why quick_take is never used as evidence")

# (c) explanation-only encoders (~15-25 min each on T4)
for mk in ["BERT-base", "RoBERTa-base"]:
    run_encoder(mk, "evid_only", seed=SEED, weighted=True)

reg = load_registry(ENC_RESULTS)
probe_keys = [f"{mk} [evid_only] seed{SEED}" for mk in ["BERT-base", "RoBERTa-base"]]
t5 = result_table([("Keyword probe (rule)", kw)] + [(k, reg[k]) for k in probe_keys] +
                  [(f"{mk} [claim_evid] (for comparison)", reg[f"{mk} [claim_evid] seed{SEED}"])
                   for mk in ["BERT-base", "RoBERTa-base"] if f"{mk} [claim_evid] seed{SEED}" in reg])
print(t5.to_string(index=False))
t5.to_csv(f"{ANA}/table_leakage_probes.csv", index=False)
print("\ninterpretation: gap between [evid_only] and [claim_evid] = the contribution of the")
print("claim-evidence interaction beyond whatever the explanation alone gives away.")

keyword probe [doctor_explanation]: acc=0.434 macro-F1=0.289
  (§5.3's archived 0.481/0.304 used the lost probe script's slightly different rule;
   all 24 swept variants give macro-F1 0.25-0.34 -> conclusion is rule-independent)
keyword probe [quick_take]:         acc=0.968 macro-F1=0.943   <- the verdict in disguise; this is why quick_take is never used as evidence
##############################################################
BERT-base [evid_only] seed42   (bert-base-uncased, lr=1e-05, max_len=512, bs=8x2)
##############################################################


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_

Step,Training Loss
100,2.174533
200,2.125237
300,2.019971
400,1.913616
500,1.754508
600,1.655031
700,1.521587


  acc=0.566  macro-F1=0.493  F1(F/MF/PT)=0.653/0.531/0.296
##############################################################
RoBERTa-base [evid_only] seed42   (roberta-base, lr=2e-05, max_len=512, bs=8x2)
##############################################################


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
100,2.198128
200,2.019355
300,1.745936
400,1.463551
500,1.246600
600,1.030534
700,0.802110


  acc=0.680  macro-F1=0.565  F1(F/MF/PT)=0.774/0.616/0.305
                                    System    Acc  Macro-F1  F1 False  F1 MostlyFalse  F1 PartlyTrue
                      Keyword probe (rule) 0.4345    0.2892    0.6069          0.1750         0.0857
              BERT-base [evid_only] seed42 0.5655    0.4934    0.6530          0.5311         0.2963
           RoBERTa-base [evid_only] seed42 0.6796    0.5652    0.7740          0.6164         0.3051
   BERT-base [claim_evid] (for comparison) 0.6165    0.5721    0.6893          0.5636         0.4634
RoBERTa-base [claim_evid] (for comparison) 0.6845    0.5768    0.7817          0.6039         0.3448

interpretation: gap between [evid_only] and [claim_evid] = the contribution of the
claim-evidence interaction beyond whatever the explanation alone gives away.


In [ ]:
%pip install openai

  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 2.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 2.3 MB/s  0:00:00 eta 0:00:01
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [openai]2m7/8 [openai]c]-types]
Note: you may need to restart the kernel to use updated packages.


## Part 6 · LLM engine ✅ setup: §5.2 (adapted from `run_llm_icl_v2.py` v2.1 → notebook v3)

**Final model set — 3 models via 2 APIs** (the fourth commercial API model was removed from the project in v4; this notebook matches the reported set, so *do not* push the old v2.1 script to the repo):

- **DeepSeek-V4-Pro** — DeepSeek API, id `deepseek-v4-pro`, thinking mode explicitly **disabled** (on by default; would swallow the token budget). Old `deepseek-chat` / `deepseek-reasoner` aliases are retired — never reuse the v1 names.
- **GPT-OSS-20B** — OpenRouter `openai/gpt-oss-20b`, low reasoning effort, 1024-token budget; final answer read from `.content` with a reasoning-trace tail fallback.
- **Gemma-4-26B-A4B** — OpenRouter `google/gemma-4-26b-a4b-it`.

In [ ]:
# ---- Part 6 · LLM engine -----------------------------------------------------
DEEPSEEK_KEY_PASTE   = ""
OPENROUTER_KEY_PASTE = ""

def get_key(name, paste):
    v = os.environ.get(name)
    if v: return v
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    return paste or ""

KEYS = {"deepseek":   get_key("DEEPSEEK_API_KEY",   DEEPSEEK_KEY_PASTE),
        "openrouter": get_key("OPENROUTER_API_KEY", OPENROUTER_KEY_PASTE)}

from openai import OpenAI
OA = {}
if KEYS["deepseek"]:
    OA["deepseek"] = OpenAI(base_url="https://api.deepseek.com", api_key=KEYS["deepseek"])
if KEYS["openrouter"]:
    OA["openrouter"] = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=KEYS["openrouter"])
for p in ("deepseek", "openrouter"):
    if not KEYS[p]:
        print(f"!! no key found for '{p}' -> its models will be SKIPPED")

# (provider, api model id, display name, max_tokens, extra request params)
LLM_MODELS = [
    ("deepseek",   "deepseek-v4-pro",           "DeepSeek-V4-Pro", 16,
        {"thinking": {"type": "disabled"}}),
    ("openrouter", "openai/gpt-oss-20b",        "GPT-OSS-20B",     1024,
        {"reasoning": {"effort": "low"}}),
    ("openrouter", "google/gemma-4-26b-a4b-it", "Gemma-4-26B-A4B", 64, None),
]

FEWSHOT_K_PER_CLASS = 2
EVID_MAX_WORDS, EVID_FEWSHOT_MAX_WORDS = 700, 200
SLEEP = {"deepseek": 0.15, "openrouter": 0.25}

SYSTEM = (
    "You are a medical fact-checking assistant. Given a health claim, classify how "
    "accurate it is into EXACTLY one of these three categories:\n"
    "- False: the claim is entirely false or has no factual basis.\n"
    "- Mostly False: largely false but with a small element of truth.\n"
    "- Partly True: partially true / half-true / misleading (a mix of accurate and inaccurate).\n"
    "Respond with ONLY the category name: False, Mostly False, or Partly True. No explanation."
)

def trunc(txt, n_words):
    return " ".join(str(txt).split()[:n_words])

def user_msg(claim, evidence):
    s = f"Claim: {claim}\n"
    if evidence:
        s += f"Evidence (doctor's explanation): {evidence}\n"
    return s + "Category:"

def build_fewshot(with_evidence):
    # fixed, balanced exemplars from TRAIN (never test), seed 42 — identical across models
    few = []
    for lab in LABELS:
        ex = train[train.label == lab].sample(FEWSHOT_K_PER_CLASS, random_state=42)
        for _, row in ex.iterrows():
            ev = trunc(row["doctor_explanation"], EVID_FEWSHOT_MAX_WORDS) if with_evidence else None
            few.append((str(row["claim_text"]), ev, lab))
    return few

def build_messages(claim, evidence, few):
    msgs = [{"role": "system", "content": SYSTEM}]
    if few:
        for c, ev, lab in few:
            msgs.append({"role": "user", "content": user_msg(c, ev)})
            msgs.append({"role": "assistant", "content": lab})
    msgs.append({"role": "user", "content": user_msg(claim, evidence)})
    return msgs

def extract_text(m):
    # final answer; falls back to the tail of the reasoning trace if content is empty
    out = (m.content or "").strip()
    if out:
        return out
    rz = getattr(m, "reasoning", None)
    if not isinstance(rz, str) or not rz:
        extra = getattr(m, "model_extra", None) or {}
        rz = extra.get("reasoning") or extra.get("reasoning_content") or ""
    return str(rz)[-400:]

def call_llm(provider, model, messages, max_toks, extra, retries=5):
    for i in range(retries):
        try:
            t0 = time.perf_counter()
            kw = dict(model=model, messages=messages, temperature=0, max_tokens=max_toks)
            if extra:
                kw["extra_body"] = extra
            r = OA[provider].chat.completions.create(**kw)
            lat = time.perf_counter() - t0
            u = getattr(r, "usage", None)
            pt = int(getattr(u, "prompt_tokens", 0) or 0)
            ct = int(getattr(u, "completion_tokens", 0) or 0)
            return extract_text(r.choices[0].message), lat, pt, ct
        except Exception as e:
            if i == retries - 1:
                raise
            wait = 3 * (i + 1)
            print(f"    retry {i+1} in {wait}s  ({type(e).__name__}: {str(e)[:90]})")
            time.sleep(wait)

def llm_run(provider, model, pretty, max_toks, extra, use_fewshot, with_evidence, limit=None):
    res_path = f"{BASE}/results_llm_evidence.json" if with_evidence else f"{BASE}/results_llm.json"
    results = load_registry(res_path)
    setting = ("few_shot" if use_fewshot else "zero_shot") + ("+evid" if with_evidence else "")
    tag = setting + (f"_n{limit}" if limit else "")
    key = f"{pretty} [{tag}]"
    tf = test.iloc[:limit] if limit else test
    if (not limit) and key in results and "error" not in results.get(key, {}):
        print(f"-- {key} already done, skipping")
        return results[key]
    few = build_fewshot(with_evidence) if use_fewshot else None
    print("#" * 62); print(f"{key}   ({provider}: {model})"); print("#" * 62)
    preds, rows, unparsed, lats, ptok, ctok = [], [], 0, [], 0, 0
    for j, (idx, row) in enumerate(tf.iterrows()):
        ev = trunc(row["doctor_explanation"], EVID_MAX_WORDS) if with_evidence else None
        out, lat, pt, ct = call_llm(provider, model,
                                    build_messages(str(row["claim_text"]), ev, few),
                                    max_toks, extra)
        lats.append(lat); ptok += pt; ctok += ct
        lab = parse_label(out)
        if lab is None:
            unparsed += 1
            lab = 0                      # score unparsed as majority class, per protocol
            if unparsed <= 5:
                print(f"    UNPARSED raw[{idx}]: {(out or '')[:120]!r}")
        preds.append(lab)
        rows.append({"row": idx, "gold": row["label"], "pred": LABELS[lab],
                     "raw": (out or "")[:150].replace("\n", " ")})
        time.sleep(SLEEP.get(provider, 0.2))
        if (j + 1) % 50 == 0:
            print(f"    {j+1}/{len(tf)}")
    res = metrics_from_preds(tf["y"].values, np.array(preds))
    res.update({"unparsed": unparsed,
                "mean_latency_s": round(float(np.mean(lats)), 3),
                "prompt_tokens": ptok, "completion_tokens": ctok})
    pd.DataFrame(rows).to_csv(f"{BASE}/preds_{safe_name(pretty + '_' + tag)}.csv", index=False)
    results = load_registry(res_path); results[key] = res; save_registry(results, res_path)
    print(f"  acc={res['acc']:.3f}  macro-F1={res['macro_f1']:.3f}  F1(F/MF/PT)="
          f"{res['f1_false']:.3f}/{res['f1_mostly_false']:.3f}/{res['f1_partly_true']:.3f}"
          f"  unparsed={res['unparsed']}  mean_lat={res['mean_latency_s']}s")
    return res

def llm_summary(res_path, title):
    results = load_registry(res_path)
    print("=" * 96); print(title); print("=" * 96)
    print(f"{'model [setting]':40s}{'n':>5s}{'acc':>8s}{'macroF1':>9s}{'F1_F':>7s}{'F1_MF':>7s}{'F1_PT':>7s}{'unp':>5s}{'lat_s':>7s}")
    for k, v in results.items():
        if "error" in v:
            print(f"{k:40s}   ERROR: {v['error'][:50]}"); continue
        if "_n" in k:
            continue                     # hide smoke-test entries
        print(f"{k:40s}{v['n']:5d}{v['acc']:8.3f}{v['macro_f1']:9.3f}"
              f"{v['f1_false']:7.3f}{v['f1_mostly_false']:7.3f}{v['f1_partly_true']:7.3f}"
              f"{v['unparsed']:5d}{v.get('mean_latency_s', float('nan')):7.2f}")

## Part 7 · LLM claim-only ICL ✅

In [ ]:
# ---- Part 7a · smoke test (optional) -----------------------------------------
if RUN_SMOKE:
    for provider, model, pretty, mt, extra in LLM_MODELS:
        if not KEYS.get(provider): continue
        for fs in (False, True):
            llm_run(provider, model, pretty, mt, extra, use_fewshot=fs, with_evidence=False, limit=25)
    print("\nsmoke complete — check every run shows unparsed=0, then proceed to the full runs.")
else:
    print("skipped (RUN_SMOKE=False)")

skipped (RUN_SMOKE=False)


In [ ]:
# ---- Part 7b · full claim-only runs (6 runs x 412 calls) ---------------------
assert len(test) == 412 and int((test["label"] == "False").sum()) == 228, \
    "TRIPWIRE: the frozen split was modified after Part 1 (e.g. a pasted train_test_split) — restore Parts 0-2 and re-run them unedited"
for provider, model, pretty, mt, extra in LLM_MODELS:
    if not KEYS.get(provider): continue
    for fs in (False, True):
        llm_run(provider, model, pretty, mt, extra, use_fewshot=fs, with_evidence=False)

llm_summary(f"{BASE}/results_llm.json", "SUMMARY — LLM in-context, claim only (§5.2)")

##############################################################
DeepSeek-V4-Pro [zero_shot]   (deepseek: deepseek-v4-pro)
##############################################################
    50/412
    100/412
    150/412
    200/412
    250/412
    300/412
    350/412
    400/412
  acc=0.551  macro-F1=0.345  F1(F/MF/PT)=0.749/0.014/0.274  unparsed=0  mean_lat=0.816s
##############################################################
DeepSeek-V4-Pro [few_shot]   (deepseek: deepseek-v4-pro)
##############################################################
    50/412
    100/412
    150/412
    200/412
    250/412
    300/412
    350/412
    400/412
  acc=0.529  macro-F1=0.399  F1(F/MF/PT)=0.721/0.203/0.272  unparsed=0  mean_lat=0.877s
##############################################################
GPT-OSS-20B [zero_shot]   (openrouter: openai/gpt-oss-20b)
##############################################################
    50/412
    100/412
    150/412
    200/412
    250/412
    300/412
    350/412

## Part 8 · LLM + evidence ✅

In [ ]:
# ---- Part 8 · full +evidence runs (6 runs x 412 calls) -----------------------
assert len(test) == 412 and int((test["label"] == "False").sum()) == 228, \
    "TRIPWIRE: the frozen split was modified after Part 1 (e.g. a pasted train_test_split) — restore Parts 0-2 and re-run them unedited"
for provider, model, pretty, mt, extra in LLM_MODELS:
    if not KEYS.get(provider): continue
    for fs in (False, True):
        llm_run(provider, model, pretty, mt, extra, use_fewshot=fs, with_evidence=True)

llm_summary(f"{BASE}/results_llm_evidence.json", "SUMMARY — LLM in-context, claim + doctor explanation (§5.2b)")

# evidence deltas (+evid minus claim-only), same model & prompt setting — §5.2b table
r0 = load_registry(f"{BASE}/results_llm.json")
r1 = load_registry(f"{BASE}/results_llm_evidence.json")
rows = []
for k1, v1 in r1.items():
    if "error" in v1 or "_n" in k1: continue
    k0 = k1.replace("+evid", "")
    if k0 in r0 and "error" not in r0[k0]:
        v0 = r0[k0]
        rows.append({"Model [setting]": k0,
                     "d_Acc": round(v1["acc"] - v0["acc"], 3),
                     "d_MacroF1": round(v1["macro_f1"] - v0["macro_f1"], 3),
                     "d_F1_MF": round(v1["f1_mostly_false"] - v0["f1_mostly_false"], 3),
                     "d_F1_PT": round(v1["f1_partly_true"] - v0["f1_partly_true"], 3)})
dt = pd.DataFrame(rows)
print("\nevidence deltas (+evid - claim-only):")
print(dt.to_string(index=False))
dt.to_csv(f"{ANA}/table_llm_evidence_deltas.csv", index=False)

##############################################################
DeepSeek-V4-Pro [zero_shot+evid]   (deepseek: deepseek-v4-pro)
##############################################################
    50/412
    100/412
    150/412
    200/412
    250/412
    300/412
    350/412
    400/412
  acc=0.624  macro-F1=0.493  F1(F/MF/PT)=0.799/0.251/0.427  unparsed=0  mean_lat=1.01s
##############################################################
DeepSeek-V4-Pro [few_shot+evid]   (deepseek: deepseek-v4-pro)
##############################################################
    50/412
    100/412
    150/412
    200/412
    250/412
    300/412
    350/412
    400/412
  acc=0.682  macro-F1=0.573  F1(F/MF/PT)=0.797/0.489/0.433  unparsed=0  mean_lat=0.89s
##############################################################
GPT-OSS-20B [zero_shot+evid]   (openrouter: openai/gpt-oss-20b)
##############################################################
    50/412
    100/412
    150/412
    200/412
    250/412
    300/41

## Part 9 · Consolidated tables + reproduction check ✅

In [ ]:
# ---- Part 9 · consolidation + fresh-vs-v4 diff -------------------------------
assert len(test) == 412 and int((test["label"] == "False").sum()) == 228, \
    "TRIPWIRE: the frozen split was modified after Part 1 (e.g. a pasted train_test_split) — restore Parts 0-2 and re-run them unedited"
enc_reg = load_registry(ENC_RESULTS)
llm0 = {k: v for k, v in load_registry(f"{BASE}/results_llm.json").items() if "error" not in v and "_n" not in k}
llm1 = {k: v for k, v in load_registry(f"{BASE}/results_llm_evidence.json").items() if "error" not in v and "_n" not in k}

# --- §5.2c headline table -----------------------------------------------------
entries = [("Majority floor", FLOOR)]
if "Keyword probe [explanation]" in enc_reg:
    entries.append(("Keyword probe (explanation)", enc_reg["Keyword probe [explanation]"]))
for k in [f"BERT-base [claim] seed{SEED}"]:
    if k in enc_reg: entries.append((k, enc_reg[k]))
if llm0:
    bk = max(llm0, key=lambda k: llm0[k]["macro_f1"])
    entries.append((f"{bk}  (best claim-only LLM)", llm0[bk]))
for k in [f"BERT-base [claim_evid] seed{SEED}", f"RoBERTa-base [claim_evid] seed{SEED}"]:
    if k in enc_reg: entries.append((k, enc_reg[k]))
if "DeepSeek-V4-Pro [few_shot+evid]" in llm1:
    entries.append(("DeepSeek-V4-Pro [few_shot+evid]", llm1["DeepSeek-V4-Pro [few_shot+evid]"]))
t9 = result_table(entries)
print("§5.2c headline table:"); print(t9.to_string(index=False))
t9.to_csv(f"{ANA}/table_headline_5_2c.csv", index=False)

# full grids for the appendix
result_table(sorted(((k, v) for k, v in enc_reg.items() if "cm" in v), key=lambda kv: kv[0])
             ).to_csv(f"{ANA}/table_all_encoders.csv", index=False)
result_table(sorted(llm0.items())).to_csv(f"{ANA}/table_all_llm_claim.csv", index=False)
result_table(sorted(llm1.items())).to_csv(f"{ANA}/table_all_llm_evid.csv", index=False)
print(f"\nfull grids saved to {ANA}/  (encoders / llm_claim / llm_evid)")

# --- fresh vs archived v4 numbers (acc, macro_f1) ----------------------------
ARCHIVED_V4 = {
    "BERT-base [claim] seed42": (0.561, 0.461),    "RoBERTa-base [claim] seed42": (0.553, 0.439),
    "BioBERT [claim] seed42": (0.541, 0.424),      "MuRIL [claim] seed42": (0.563, 0.415),
    "BERT-base [claim_evid] seed42": (0.619, 0.564), "RoBERTa-base [claim_evid] seed42": (0.694, 0.574),
    "BioBERT [claim_evid] seed42": (0.619, 0.542), "MuRIL [claim_evid] seed42": (0.585, 0.410),
    # keyword probe intentionally absent: its original rule was lost with the probe script
    # (see Part 5 provenance note) — the fresh protocol-consistent probe is now canonical
    "DeepSeek-V4-Pro [zero_shot]": (0.5558, 0.3518), "DeepSeek-V4-Pro [few_shot]": (0.5170, 0.3812),
    "GPT-OSS-20B [zero_shot]": (0.5655, 0.4287),     "GPT-OSS-20B [few_shot]": (0.5243, 0.3738),
    "Gemma-4-26B-A4B [zero_shot]": (0.4927, 0.3240), "Gemma-4-26B-A4B [few_shot]": (0.4927, 0.3491),
    "DeepSeek-V4-Pro [zero_shot+evid]": (0.6262, 0.4869), "DeepSeek-V4-Pro [few_shot+evid]": (0.6772, 0.5794),
    "GPT-OSS-20B [zero_shot+evid]": (0.5995, 0.4284),     "GPT-OSS-20B [few_shot+evid]": (0.6165, 0.4181),
    "Gemma-4-26B-A4B [zero_shot+evid]": (0.6408, 0.5174), "Gemma-4-26B-A4B [few_shot+evid]": (0.6359, 0.4966),
}
ALL_NOW = {**enc_reg, **llm0, **llm1}
print("\nreproduction diff vs lab-notebook v4 (fresh - archived):")
mx = 0.0
for k, (a_acc, a_f1) in ARCHIVED_V4.items():
    if k not in ALL_NOW:
        print(f"  [ -- ] {k}: not run yet"); continue
    da, dfi = ALL_NOW[k]["acc"] - a_acc, ALL_NOW[k]["macro_f1"] - a_f1
    mx = max(mx, abs(da), abs(dfi))
    print(f"  [{'OK ' if max(abs(da), abs(dfi)) < 0.02 else '>2%'}] {k:42s} dAcc={da:+.4f}  dMacroF1={dfi:+.4f}")
print(f"\nmax |delta| across completed runs: {mx:.4f}"
      "   (== the run-to-run stability number for the paper, if this was a fresh rerun)")

§5.2c headline table:
                                        System    Acc  Macro-F1  F1 False  F1 MostlyFalse  F1 PartlyTrue
                                Majority floor 0.5534    0.2375    0.7125          0.0000         0.0000
                   Keyword probe (explanation) 0.4345    0.2892    0.6069          0.1750         0.0857
                      BERT-base [claim] seed42 0.5437    0.4486    0.6429          0.4675         0.2353
GPT-OSS-20B [zero_shot]  (best claim-only LLM) 0.5607    0.4408    0.6992          0.3529         0.2703
                 BERT-base [claim_evid] seed42 0.6165    0.5721    0.6893          0.5636         0.4634
              RoBERTa-base [claim_evid] seed42 0.6845    0.5768    0.7817          0.6039         0.3448
               DeepSeek-V4-Pro [few_shot+evid] 0.6820    0.5732    0.7970          0.4895         0.4333

full grids saved to outputs/analysis/  (encoders / llm_claim / llm_evid)

reproduction diff vs lab-notebook v4 (fresh - archived):
  [OK 

## Part 10 · Bootstrap CIs ⬜→✅

In [ ]:
# ---- Part 10 · bootstrap -----------------------------------------------------
assert len(test) == 412 and int((test["label"] == "False").sum()) == 228, \
    "TRIPWIRE: the frozen split was modified after Part 1 (e.g. a pasted train_test_split) — restore Parts 0-2 and re-run them unedited"
# (preds_path_for / load_pred_vec / HEADLINE_KEYS live in the Part-2 engine)
HEADLINE = [k for k in HEADLINE_KEYS if os.path.exists(preds_path_for(k))]
print("headline systems with predictions on disk:", len(HEADLINE))

N_BOOT = 1000
rng = np.random.default_rng(42)
IDX = rng.integers(0, len(GOLD), size=(N_BOOT, len(GOLD)))   # shared resamples -> paired stats

def boot_stats(pred):
    out = np.empty((N_BOOT, 3))
    for b in range(N_BOOT):
        g, p = GOLD[IDX[b]], pred[IDX[b]]
        out[b, 0] = accuracy_score(g, p)
        out[b, 1] = f1_score(g, p, average="macro", zero_division=0)
        out[b, 2] = f1_score(g, p, labels=[2], average=None, zero_division=0)[0]
    return out

BOOT = {k: boot_stats(load_pred_vec(k)) for k in HEADLINE}
rows = []
for k in HEADLINE:
    point = metrics_from_preds(GOLD, load_pred_vec(k))
    lo, hi = np.percentile(BOOT[k], [2.5, 97.5], axis=0)
    rows.append({"System": k,
                 "Acc [95% CI]":     f"{point['acc']:.3f} [{lo[0]:.3f}, {hi[0]:.3f}]",
                 "MacroF1 [95% CI]": f"{point['macro_f1']:.3f} [{lo[1]:.3f}, {hi[1]:.3f}]",
                 "PT-F1 [95% CI]":   f"{point['f1_partly_true']:.3f} [{lo[2]:.3f}, {hi[2]:.3f}]"})
t10 = pd.DataFrame(rows)
print(t10.to_string(index=False))
t10.to_csv(f"{ANA}/table_bootstrap_cis.csv", index=False)

print("\npaired macro-F1 deltas (A - B, same resamples):")
for a, b in [("DeepSeek-V4-Pro [few_shot+evid]", f"RoBERTa-base [claim_evid] seed{SEED}"),
             (f"RoBERTa-base [claim_evid] seed{SEED}", f"BERT-base [claim_evid] seed{SEED}")]:
    if a in BOOT and b in BOOT:
        d = BOOT[a][:, 1] - BOOT[b][:, 1]
        lo, hi = np.percentile(d, [2.5, 97.5])
        verdict = "CI crosses 0 -> parity language holds" if lo < 0 < hi else "CI excludes 0 -> difference is real"
        print(f"  {a}  -  {b}\n    point={d.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]   {verdict}")

headline systems with predictions on disk: 6
                          System         Acc [95% CI]     MacroF1 [95% CI]       PT-F1 [95% CI]
        BERT-base [claim] seed42 0.544 [0.495, 0.590] 0.449 [0.392, 0.503] 0.235 [0.103, 0.356]
         GPT-OSS-20B [zero_shot] 0.561 [0.515, 0.607] 0.441 [0.379, 0.495] 0.270 [0.131, 0.400]
   BERT-base [claim_evid] seed42 0.617 [0.573, 0.663] 0.572 [0.514, 0.630] 0.463 [0.329, 0.585]
RoBERTa-base [claim_evid] seed42 0.684 [0.641, 0.728] 0.577 [0.510, 0.637] 0.345 [0.186, 0.492]
Gemma-4-26B-A4B [zero_shot+evid] 0.638 [0.592, 0.677] 0.512 [0.455, 0.563] 0.400 [0.283, 0.504]
 DeepSeek-V4-Pro [few_shot+evid] 0.682 [0.638, 0.723] 0.573 [0.503, 0.636] 0.433 [0.255, 0.582]

paired macro-F1 deltas (A - B, same resamples):
  DeepSeek-V4-Pro [few_shot+evid]  -  RoBERTa-base [claim_evid] seed42
    point=-0.0038  95% CI [-0.0853, +0.0826]   CI crosses 0 -> parity language holds
  RoBERTa-base [claim_evid] seed42  -  BERT-base [claim_evid] seed42
    point

In [ ]:
%pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 15.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]
Note: you may need to restart the kernel to use updated packages.


## Part 11 · McNemar tests ⬜→✅

In [ ]:
# ---- Part 11 · McNemar -------------------------------------------------------
from statsmodels.stats.contingency_tables import mcnemar

def mcnemar_pair(keyA, keyB):
    a = load_pred_vec(keyA) == GOLD
    b = load_pred_vec(keyB) == GOLD
    n11, n10 = int(np.sum(a & b)), int(np.sum(a & ~b))
    n01, n00 = int(np.sum(~a & b)), int(np.sum(~a & ~b))
    res = mcnemar([[n11, n10], [n01, n00]], exact=True)
    print(f"{keyA}\n  vs {keyB}")
    print(f"  acc A={a.mean():.4f}  acc B={b.mean():.4f}  |  discordant: A-only-correct={n10}, B-only-correct={n01}")
    print(f"  exact McNemar p = {res.pvalue:.4f}   -> "
          f"{'no significant accuracy difference' if res.pvalue >= 0.05 else 'significant accuracy difference'} at alpha=0.05\n")

mcnemar_pair("DeepSeek-V4-Pro [few_shot+evid]", f"RoBERTa-base [claim_evid] seed{SEED}")
mcnemar_pair(f"RoBERTa-base [claim_evid] seed{SEED}", f"BERT-base [claim_evid] seed{SEED}")
print("(McNemar = paired accuracy; for macro-F1 use the paired bootstrap deltas from Part 10.)")

DeepSeek-V4-Pro [few_shot+evid]
  vs RoBERTa-base [claim_evid] seed42
  acc A=0.6820  acc B=0.6845  |  discordant: A-only-correct=61, B-only-correct=62
  exact McNemar p = 1.0000   -> no significant accuracy difference at alpha=0.05

RoBERTa-base [claim_evid] seed42
  vs BERT-base [claim_evid] seed42
  acc A=0.6845  acc B=0.6165  |  discordant: A-only-correct=68, B-only-correct=40
  exact McNemar p = 0.0091   -> significant accuracy difference at alpha=0.05

(McNemar = paired accuracy; for macro-F1 use the paired bootstrap deltas from Part 10.)


## Part 12 · Multi-seed ⬜→✅

In [ ]:
# ---- Part 12 · multi-seed ----------------------------------------------------
assert len(test) == 412 and int((test["label"] == "False").sum()) == 228, \
    "TRIPWIRE: the frozen split was modified after Part 1 (e.g. a pasted train_test_split) — restore Parts 0-2 and re-run them unedited"
for mk, cond in MULTISEED_CONFIGS:
    for s in SEEDS_MULTI:
        run_encoder(mk, cond, seed=s, weighted=True)

reg = load_registry(ENC_RESULTS)
rows = []
for mk, cond in MULTISEED_CONFIGS:
    got = [reg[f"{mk} [{cond}] seed{s}"] for s in SEEDS_MULTI if f"{mk} [{cond}] seed{s}" in reg]
    if not got: continue
    for metric, nice in [("acc", "Acc"), ("macro_f1", "Macro-F1"), ("f1_partly_true", "PT-F1")]:
        vals = np.array([g[metric] for g in got])
        rows.append({"Config": f"{mk} [{cond}]", "Metric": nice, "n_seeds": len(vals),
                     "Mean": round(float(vals.mean()), 4), "Std": round(float(vals.std(ddof=1)), 4),
                     "Values": ", ".join(f"{v:.3f}" for v in vals)})
t12 = pd.DataFrame(rows)
print(t12.to_string(index=False))
t12.to_csv(f"{ANA}/table_multiseed.csv", index=False)

-- BERT-base [claim] seed42 already done, skipping
##############################################################
BERT-base [claim] seed43   (bert-base-uncased, lr=1e-05, max_len=256, bs=16x1)
##############################################################


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_

Step,Training Loss
100,1.105148
200,1.056036
300,1.018204
400,0.912597
500,0.869349
600,0.772015
700,0.729809


  acc=0.485  macro-F1=0.399  F1(F/MF/PT)=0.598/0.444/0.155
##############################################################
BERT-base [claim] seed44   (bert-base-uncased, lr=1e-05, max_len=256, bs=16x1)
##############################################################


...

  acc=0.660  macro-F1=0.534  F1(F/MF/PT)=0.773/0.552/0.276
##############################################################
RoBERTa-base [claim_evid] seed44   (roberta-base, lr=2e-05, max_len=512, bs=8x2)
##############################################################


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
100,2.207203
200,2.098718
300,1.891951
400,1.591912
500,1.332132
600,1.198694
700,0.949810


  acc=0.689  macro-F1=0.566  F1(F/MF/PT)=0.786/0.620/0.291
                   Config   Metric  n_seeds   Mean    Std              Values
        BERT-base [claim]      Acc        3 0.5048 0.0337 0.544, 0.485, 0.485
        BERT-base [claim] Macro-F1        3 0.4121 0.0321 0.449, 0.399, 0.389
        BERT-base [claim]    PT-F1        3 0.1773 0.0508 0.235, 0.155, 0.141
   BERT-base [claim_evid]      Acc        3 0.6165 0.0146 0.617, 0.631, 0.602
   BERT-base [claim_evid] Macro-F1        3 0.5458 0.0413 0.572, 0.567, 0.498
   BERT-base [claim_evid]    PT-F1        3 0.3883 0.1199 0.463, 0.452, 0.250
RoBERTa-base [claim_evid]      Acc        3 0.6780 0.0156 0.684, 0.660, 0.689
RoBERTa-base [claim_evid] Macro-F1        3 0.5587 0.0224 0.577, 0.534, 0.566
RoBERTa-base [claim_evid]    PT-F1        3 0.3039 0.0362 0.345, 0.276, 0.291


## Part 13 · Error analysis — Hooks 1–4 ⬜→✅

In [ ]:
# ---- Part 13a · Hooks 1-2 ----------------------------------------------------
assert len(test) == 412 and int((test["label"] == "False").sum()) == 228, \
    "TRIPWIRE: the frozen split was modified after Part 1 (e.g. a pasted train_test_split) — restore Parts 0-2 and re-run them unedited"
enc_reg = load_registry(ENC_RESULTS)
llm0 = {k: v for k, v in load_registry(f"{BASE}/results_llm.json").items() if "error" not in v and "_n" not in k}
llm1 = {k: v for k, v in load_registry(f"{BASE}/results_llm_evidence.json").items() if "error" not in v and "_n" not in k}
ALL20 = ([f"{mk} [claim] seed{SEED}" for mk in ENCODERS] +
         [f"{mk} [claim_evid] seed{SEED}" for mk in ENCODERS] +
         list(llm0.keys()) + list(llm1.keys()))
P = {k: load_pred_vec(k) for k in ALL20 if os.path.exists(preds_path_for(k))}
print(f"loaded per-item predictions for {len(P)}/20 reported runs\n")

mF, mMF, mPT = GOLD == 0, GOLD == 1, GOLD == 2
rows = []
for k, pv in P.items():
    rows.append({"Run": k, "Acc": round(float((pv == GOLD).mean()), 3),
                 "F_recall": round(float((pv[mF] == 0).mean()), 3),
                 "MF->F %": round(float((pv[mMF] == 0).mean()) * 100, 1),
                 "PT->F %": round(float((pv[mPT] == 0).mean()) * 100, 1)})
t13a = pd.DataFrame(rows).sort_values("Run")
print("Hook 1 — severity collapse per run:"); print(t13a.to_string(index=False))
t13a.to_csv(f"{ANA}/table_hook1_severity_collapse.csv", index=False)

pairs = [(k.replace("+evid", ""), k) for k in P if "+evid" in k] + \
        [(f"{mk} [claim] seed{SEED}", f"{mk} [claim_evid] seed{SEED}") for mk in ENCODERS]
rows = []
for k0, k1 in pairs:
    if k0 in P and k1 in P:
        rows.append({"Pair": k0, "MF->F claim %": round(float((P[k0][mMF] == 0).mean()) * 100, 1),
                     "MF->F +evid %": round(float((P[k1][mMF] == 0).mean()) * 100, 1),
                     "PT->F claim %": round(float((P[k0][mPT] == 0).mean()) * 100, 1),
                     "PT->F +evid %": round(float((P[k1][mPT] == 0).mean()) * 100, 1)})
t13b = pd.DataFrame(rows)
print("\nHook 2 — what evidence fixes (collapse rates, claim vs +evid):")
print(t13b.to_string(index=False))
t13b.to_csv(f"{ANA}/table_hook2_evidence_fixes.csv", index=False)

loaded per-item predictions for 20/20 reported runs

Hook 1 — severity collapse per run:
                             Run   Acc  F_recall  MF->F %  PT->F %
        BERT-base [claim] seed42 0.544     0.632     41.4     41.0
   BERT-base [claim_evid] seed42 0.617     0.623     25.5     12.8
          BioBERT [claim] seed42 0.544     0.592     37.2     30.8
     BioBERT [claim_evid] seed42 0.612     0.654     31.0     15.4
 DeepSeek-V4-Pro [few_shot+evid] 0.682     0.921     54.5     25.6
      DeepSeek-V4-Pro [few_shot] 0.529     0.794     53.8     38.5
DeepSeek-V4-Pro [zero_shot+evid] 0.624     0.908     51.0     23.1
     DeepSeek-V4-Pro [zero_shot] 0.551     0.921     69.0     59.0
     GPT-OSS-20B [few_shot+evid] 0.609     0.969     77.9     53.8
          GPT-OSS-20B [few_shot] 0.524     0.754     55.9     59.0
    GPT-OSS-20B [zero_shot+evid] 0.600     0.934     71.7     43.6
         GPT-OSS-20B [zero_shot] 0.561     0.785     59.3     48.7
 Gemma-4-26B-A4B [few_shot+evid] 0.621  

In [ ]:
# ---- Part 13b · Hook 3 consensus errors --------------------------------------
M = np.vstack([P[k] for k in P])
wrong_all = np.all(M != GOLD, axis=0)
head_avail = [k for k in HEADLINE_KEYS if k in P]
wrong_head = np.all(np.vstack([P[k] != GOLD for k in head_avail]), axis=0) if head_avail else wrong_all

cons = test.iloc[np.where(wrong_all)[0]][
    ["id", "claim_text", "label", "categories", "traditional_medicine_involved",
     "health_harm_potential", "target_demographic"]].copy()
cons.insert(0, "row", test.index[np.where(wrong_all)[0]])
for k in head_avail:
    cons[f"pred: {k}"] = [LABELS[p] for p in P[k][wrong_all]]
cons.to_csv(f"{ANA}/consensus_errors.csv", index=False)
print(f"Hook 3 — wrong in ALL {len(P)} runs: {int(wrong_all.sum())} items"
      f"  |  wrong in all {len(head_avail)} headline systems: {int(wrong_head.sum())} items")
print(f"saved -> {ANA}/consensus_errors.csv   (feeds the Phase-2 targeted sample + qualitative writeup)\n")
print("examples (gold label | claim):")
for _, r in cons.head(5).iterrows():
    print(f"  [{r['label']:>12s}] {str(r['claim_text'])[:100]}")

Hook 3 — wrong in ALL 20 runs: 11 items  |  wrong in all 6 headline systems: 28 items
saved -> outputs/analysis/consensus_errors.csv   (feeds the Phase-2 targeted sample + qualitative writeup)

examples (gold label | claim):
  [Mostly False] Fabiflu 200 mg. has been priced at Rs.103 per tablet by Glenmark. After my inquest, I found, already
  [Mostly False] The People’s Voice article from June 29, 2023 falsely alleges that the doctors in the Netherlands ar
  [       False] Drinking a concoction of ginger and clove can help in weight loss.
  [ Partly True] Hospitals will be empty when everyone knows that a single cup is enough to cleanse the liver and ski
  [Mostly False] Russian President Vladimir Putin has ordered the destruction of all COVID vaccines in Russia


In [ ]:
# ---- Part 13c · Hook 4 error rate x metadata ---------------------------------
# (multi() lives in the Part-2 engine — same parser as audit_indicmedfacts.py)
for sysk in ["DeepSeek-V4-Pro [few_shot+evid]", f"RoBERTa-base [claim_evid] seed{SEED}"]:
    if sysk not in P: continue
    sub = test.copy(); sub["err"] = (P[sysk] != GOLD)
    print("=" * 80); print(f"Hook 4 — {sysk}   (overall error rate {sub['err'].mean():.3f})"); print("=" * 80)
    tm = sub.groupby(sub["traditional_medicine_involved"].astype(str))["err"].agg(["mean", "count"]).round(3)
    print("by traditional-medicine flag:"); print(tm.to_string())
    hm = sub.groupby("health_harm_potential")["err"].agg(["mean", "count"]).round(3)
    print("\nby harm potential (1-5):"); print(hm.to_string())
    ex = sub[["err", "categories"]].copy()
    ex["cat"] = ex["categories"].apply(multi)
    ex = ex.explode("cat").dropna(subset=["cat"])
    top = ex.groupby("cat")["err"].agg(["mean", "count"]).query("count >= 15").sort_values("mean", ascending=False)
    print("\nby category (support >= 15 test items), worst first:"); print(top.round(3).to_string())
    hi = sub[(sub["traditional_medicine_involved"].astype(str).str.lower().isin(["true", "1", "1.0"])) &
             (sub["health_harm_potential"] >= 4)]
    print(f"\nTM-involved AND harm>=4: n={len(hi)}, error rate={hi['err'].mean():.3f}"
          if len(hi) else "\nTM-involved AND harm>=4: no test items")
    tm.to_csv(f"{ANA}/hook4_{safe_name(sysk)}_by_tm.csv")
    hm.to_csv(f"{ANA}/hook4_{safe_name(sysk)}_by_harm.csv")
    top.to_csv(f"{ANA}/hook4_{safe_name(sysk)}_by_category.csv")
    print()

Hook 4 — DeepSeek-V4-Pro [few_shot+evid]   (overall error rate 0.318)
by traditional-medicine flag:
                                mean  count
traditional_medicine_involved              
False                          0.336    232
True                           0.294    180

by harm potential (1-5):
                        mean  count
health_harm_potential              
1                      0.316     19
2                      0.331    154
3                      0.356    146
4                      0.262     80
5                      0.077     13

by category (support >= 15 test items), worst first:
                                   mean  count
cat                                           
Pharmaceuticals and Drugs         0.474     19
Brain Health                      0.467     15
Dental Health                     0.423     26
Cancer                            0.394     33
Respiratory                       0.375     16
Infectious Diseases               0.352     54
Traditional/Alte

## Part 14 · Phase-2 annotation sample builder ⬜

In [ ]:
# ---- Part 14 · build annotation sample ---------------------------------------
def strat_sample(frame, n, seed):
    shares = frame["label"].value_counts(normalize=True)
    take = {lab: int(round(shares[lab] * n)) for lab in shares.index}
    order = list(shares.index)
    i = 0
    while sum(take.values()) != n:
        take[order[i % len(order)]] += 1 if sum(take.values()) < n else -1
        i += 1
    return pd.concat([frame[frame.label == lab].sample(take[lab], random_state=seed) for lab in take])

rand150 = strat_sample(test, 150, seed=2026)
print("random-150 (from TEST, stratified):", rand150["label"].value_counts().to_dict())

# targeted pool ---------------------------------------------------------------
flag = df["traditional_medicine_involved"].astype(str).str.strip().str.lower().isin(["true", "1", "1.0", "yes"])
tmcat = df["categories"].apply(lambda x: any("tradition" in c.lower() for c in multi(x)))
tm_contra = df[~flag & tmcat]                                   # §3b: expected 11
v = df["verdict_original"].astype(str)
pl = v.apply(parse_label)                                       # engine parser on the raw verdicts
contra = pl.notna() & (pl != df["label"].map(label2id))         # verdict text disagrees with final label
vl = v.str.lower()
boundary = df[contra | vl.str.contains("partly false") | vl.str.contains("satire")]
cons_rows = []
if os.path.exists(f"{ANA}/consensus_errors.csv"):
    cons_rows = pd.read_csv(f"{ANA}/consensus_errors.csv")["row"].tolist()
else:
    print("⚠ consensus_errors.csv not found — run Part 13 first for the full targeted pool")

# priority within the boundary pool: shortest verdicts first (pure rating-language
# conflicts like 'Half True (Quick Take) / mostly false') before long incidental wording
bound_order = boundary.assign(_w=boundary["verdict_original"].astype(str).str.split().str.len()
                              ).sort_values("_w").index
targeted, reasons = [], {}
def _take(idx, reason, cap=50):
    if idx not in reasons and idx not in rand150.index and len(targeted) < cap:
        targeted.append(idx); reasons[idx] = reason
for idx in tm_contra.index:                                     # all contradictions go in
    _take(idx, "tm_contradiction")
for idx in bound_order:                                         # boundary, capped at 25
    if sum(r == "verdict_boundary" for r in reasons.values()) >= 25: break
    _take(idx, "verdict_boundary")
for idx in cons_rows:                                           # consensus errors fill next
    _take(idx, "consensus_error")
for idx in bound_order:                                         # backfill if slots remain
    _take(idx, "verdict_boundary")
print(f"targeted: {len(targeted)}  breakdown: {Counter(reasons.values())}")
if len(targeted) < 50:
    print(f"  (pool below the 50-slot budget — Hook-3 consensus errors from the real runs top it up;"
          f" re-run this cell after Part 13)")

samp = pd.concat([rand150, df.loc[targeted]])
samp = samp[~samp.index.duplicated()]
samp = samp.sample(frac=1, random_state=7)                      # shuffle presentation order
samp["sample_id"] = [f"S{i+1:03d}" for i in range(len(samp))]
samp["is_targeted"] = [i in reasons for i in samp.index]
samp["targeted_reason"] = [reasons.get(i, "random") for i in samp.index]

# Pass A: blind, claim-only
passA = samp[["sample_id", "id", "claim_text"]].copy()
passA["annotator1_label"] = ""; passA["annotator2_label"] = ""; passA["notes"] = ""
passA.to_csv(f"{ANN}/passA_claim_only_blind.csv", index=False)

# Pass B: 100-item stratified subset of the random-150, WITH the doctor explanation
passB_pool = samp[samp["targeted_reason"] == "random"]
passB = strat_sample(passB_pool, 100, seed=2027)[["sample_id", "id", "claim_text", "doctor_explanation"]].copy()
passB["annotator1_label"] = ""; passB["annotator2_label"] = ""; passB["notes"] = ""
passB.to_csv(f"{ANN}/passB_claim_plus_evidence.csv", index=False)

# key file — annotators must NOT open this until both passes are done
key = samp[["sample_id", "id", "label", "split", "is_targeted", "targeted_reason"]].copy()
key.insert(1, "row", samp.index)
key.to_csv(f"{ANN}/ANSWER_KEY_do_not_open.csv", index=False)
print(f"\nwrote {len(passA)}-item Pass-A sheet, {len(passB)}-item Pass-B sheet, and the key -> {ANN}/")
print("label set for annotators: False / Mostly False / Partly True  (definitions = the system prompt in Part 6)")

random-150 (from TEST, stratified): {'False': 83, 'Mostly False': 53, 'Partly True': 14}
targeted: 50  breakdown: Counter({'verdict_boundary': 32, 'tm_contradiction': 10, 'consensus_error': 8})

wrote 200-item Pass-A sheet, 100-item Pass-B sheet, and the key -> outputs/annotation/
label set for annotators: False / Mostly False / Partly True  (definitions = the system prompt in Part 6)


In [ ]:
# ---- Part 14b · kappa computation (run AFTER annotation) ---------------------
FILLED_A = f"{ANN}/passA_claim_only_blind.csv"          # replace with the filled files
FILLED_B = f"{ANN}/passB_claim_plus_evidence.csv"
key = pd.read_csv(f"{ANN}/ANSWER_KEY_do_not_open.csv")

for name, path in [("Pass A (claim-only)", FILLED_A), ("Pass B (claim+evidence)", FILLED_B)]:
    if not os.path.exists(path):
        print(f"{name}: file not found"); continue
    a = pd.read_csv(path).merge(key[["sample_id", "label", "is_targeted"]], on="sample_id")
    a = a.dropna(subset=["annotator1_label", "annotator2_label"])
    a = a[(a["annotator1_label"].astype(str).str.strip() != "") &
          (a["annotator2_label"].astype(str).str.strip() != "")]
    if not len(a):
        print(f"{name}: no filled rows yet — fill annotator1_label / annotator2_label and re-run"); continue
    for scope, sub in [("ALL", a), ("random-only (headline κ)", a[~a["is_targeted"]])]:
        if not len(sub): continue
        kappa = cohen_kappa_score(sub["annotator1_label"], sub["annotator2_label"])
        acc1 = (sub["annotator1_label"] == sub["label"]).mean()
        acc2 = (sub["annotator2_label"] == sub["label"]).mean()
        print(f"{name} · {scope}: n={len(sub)}  κ={kappa:.3f}  acc-vs-gold A1={acc1:.3f} A2={acc2:.3f}")
print("\nreport: random-150 κ = headline agreement; targeted-50 separately; Pass-B accuracy = human ceiling")
print("with evidence; Pass-A vs Pass-B accuracy gap = the human analogue of the model evidence delta.")

Pass A (claim-only): no filled rows yet — fill annotator1_label / annotator2_label and re-run
Pass B (claim+evidence): no filled rows yet — fill annotator1_label / annotator2_label and re-run

report: random-150 κ = headline agreement; targeted-50 separately; Pass-B accuracy = human ceiling
with evidence; Pass-A vs Pass-B accuracy gap = the human analogue of the model evidence delta.


## Part 15 · Archive ⚠

In [ ]:
# ---- Part 15 · zip outputs ---------------------------------------------------
zpath = f"{OUT_ROOT}/indicmedfacts_artifacts.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(OUT_ROOT):
        for fn in files:
            fp = os.path.join(root, fn)
            if fp == zpath: continue
            z.write(fp, os.path.relpath(fp, OUT_ROOT))
print("archived ->", zpath, f"({os.path.getsize(zpath)/1e6:.1f} MB)")
print("\nbefore pushing to GitHub: anonymized repo for blind review · no API keys ·")
print("no annotator names in the sheets · keep the archived v4 results JSONs alongside the fresh ones.")

archived -> outputs/indicmedfacts_artifacts.zip (0.2 MB)

before pushing to GitHub: anonymized repo for blind review · no API keys ·
no annotator names in the sheets · keep the archived v4 results JSONs alongside the fresh ones.
